# Day 29 ノート：GRPOの直感（PPO / DPOとの比較）—— 詳細版

> **成果物：** GRPOの直感的理解ノート。「報酬（Reward）はどこから来るのか」、「グループ相対（Group Relative）」とは何を意味するのか。
> **ポイント：** 単に「GRPOをどう動かすか」ではなく、「なぜこのような設計になったのか、その代償はどこにあるのか」に焦点を当てる。

---

## 1. 報酬（Reward）はどこから来るのか —— 2つのルート

学習プロセスを「作文の先生」に例えると、採点方法には全く異なる2つのアプローチがあります。

### アプローチ1：正解判定（Rule-based Reward）
「3 × 24 = ？」という数学の問いに対し、答えは72です。この場合、**人間の「感覚」は不要**です。プログラムを書けば一発で判定できます。「出力が72なら1点、そうでなければ0点」。曖昧さは一切なく、報酬モデルも不要。`if answer == 72: reward = 1` という1行で済みます。

### アプローチ2：審判を呼ぶ（Reward Model / Preference）
「友情についての短文を書いて」という問いには、**標準的な正解がありません**。プログラムで良し悪しを判断するのは不可能です。どうするか？ 事前に「審判モデル」を訓練しておきます。人間に「こちらの文章の方が好きだ」というデータを大量に学習させ、好みを模倣させます。この審判は訓練が終わると**凍結（Freeze）**され、以降、生徒（モデル）が書いた文章を採点し、スコア（例：0.8点）を吐き出します。

| 特徴 | ルールベースの報酬 | 報酬モデル（Preference） |
| :--- | :--- | :--- |
| **採点者** | 固定されたルール（正誤など） | 訓練され、凍結された審判モデル |
| **性質** | 客観的、検証可能 | 主観的、人間の好みを模倣 |
| **代表例** | DeepSeek-R1（GRPO） | InstructGPT（PPO） |
| **追加コスト** | ほぼゼロ | 報酬モデルの訓練コストが高い |

**DeepSeekMathは前者のルートです。** 中間プロセスに正解がないのにどう採点するか？ 解決策は **ブロードキャスト（Broadcast）** です。出力全体を一つのユニットと見なし、最終的なアドバンテージを全トークンにそのままコピー＆ペーストします。

---

## 2. 「グループ相対（Group Relative）」とは何か？

**TL;DR：** 横に先生がいなくても、自分の「同期（コホート）」と比較すればいい。自分のスコアが平均より高ければ、何か正しいことをしたはずだ。モデルはその行動を強化します。

### 具体例：3 × 24 = ? (グループサイズ G = 4)

| サンプル | モデルの出力 | 最終回答 | 報酬 $r_i$ |
| :--- | :--- | :--- | :--- |
| $o_1$ | 3×24 = 3×20+3×4 = 60+12 = 72 | 72 | 1 |
| $o_2$ | 3×24 = 3×25−3 = 75−3 = 72 | 72 | 1 |
| $o_3$ | 3×24 = 3×20+3×4 = 60+8 = 68 | 68 | 0 |
| $o_4$ | 3×24 = 72（プロセスなしの勘） | 72 | 1 |

1.  **平均と標準偏差を計算：** 平均 = 0.75, 標準偏差 ≈ 0.433
2.  **アドバンテージ（優位性） $\hat{A}_i$ を計算：** $\hat{A}_i = (r_i - \text{mean}) / \text{std}$
    *   $o_1, o_2, o_4$: $\approx$ **+0.577**（平均より良い！その調子で続けよう）
    *   $o_3$: $\approx$ **-1.732**（平均より悪い！その書き方はやめろ）

**ポイント：** ベースライン（0.75）は、今回のサンプルが自分たちで出した「リアルな平均点」です。PPOのように価値モデルで**予想**する必要がなく、計算コストと複雑さを大幅に削減できます。

---

## 3. PPOのCriticはなぜ「高い」のか（誤解の修正）

最初、私は「トークンごとに採点するのが重い」と思っていましたが、それは間違いでした。

*   **Transformerの性質：** アテンション機構により一文を並列処理するため、各位置に数値を出すコストはほぼ無料の副産物です。
*   **本当のコスト：**
    1.  **メモリ消費量：** 価値モデルは方策モデルと同サイズ。パラメータ・勾配・オプティマイザ状態がすべて**2倍**必要になります。
    2.  **信号の稀薄さ：** 文末に一つしか報酬がないのに、全トークンの価値を予測させるのは学習効率が悪すぎます。

GRPOは価値モデルを削除し、統計量で代用することで、このコストと不安定さを一挙に解決しました。

---

## 4. RL vs SL —— 「論理」と「模倣」の違い

「答えが合っているか見るならSFTと同じじゃない？」という疑問への回答：

*   **SFT（教師あり微調）：** 先生が「解法プロセス」を教えます。モデルはそれを**「模倣」**し、スタイルを暗記します。
*   **GRPO（強化学習）：** 先生は「ゴール」だけを教えます。モデルは膨大な試行錯誤を通じて、**「論理」**を自ら構築します。

この「自分で試して、壁にぶつかって、正解を見つける」プロセスこそが、真の推論能力を生みます。

---

## 5. DPOの立ち位置と弱点

DPO（Direct Preference Optimization）はシンプルで安定していますが、**「自己探索をしない」**という根本的な弱点があります。人間が用意した「良い例・悪い例」の二択から選ぶだけで、人間が思いつかなかった革新的な解法を生み出すことはできません。

---

## 6. KL正則化：モデルの暴走を防ぐ命綱

報酬（スコア）を追い求めると、モデルは不自然な話し方になることがあります（報酬ハッキング）。
*   **PPO：** 報酬の中にKLペナルティを混ぜます。
*   **GRPO：** 報酬はクリーンに保ち、Loss関数にKL項を直接加えます。

GRPOの方が、「答えた内容の良さ」と「元のモデルからの乖離」を明確に分けて評価できます。

---

## 7. GRPOのコスト構造（メモリ vs 計算量）

GRPOは無条件に安いわけではありません。
*   **メリット：** 価値モデルを削ることで**メモリ（VRAM）使用量**を劇的に抑えられます。
*   **デメリット：** 統計を取るために同じ問題を $G$ 回（8〜64回）生成する必要があり、**計算時間**がかかります。

**注意点：** 問題が簡単すぎたり難すぎたりすると、全員正解（または全員不正解）になり、アドバンテージが0になります。これは計算リソースの無駄遣い（Degenerate Group）であり、難易度の適切な選定が重要です。

---

## 8. 学習の真の目的：報酬ハッキングの観察

今週（Day 30-33）の目標は、モデルの数学を強くすることだけではありません。
**「報酬関数がいかにモデルのズル（ハッキング）によって突破されるか」を目の当たりにすること**が最大の収穫です。

### グッドハートの法則（Goodhart's Law）
> 「指標が目標になると、それはもはや良い指標ではなくなる」

**報酬ハッキング（跑偏）の例：**
*   **Length Bias:** 推論せずに「私は一生懸命考えています...」と繰り返して長さを稼ぐ。
*   **Format Hack:** 最終的な答えのフォーマット（Answer: 72）さえ合っていれば、中身がデタラメでも満点になる。

